# Deaths of Despair — Layer 1: Descriptive Pass

Automated overview of all indicators in the processed data.

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import sys
sys.path.insert(0, "/Users/matt/data-adventures")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pipeline.config import load_config, get_data_processed_dir

PROJECT = "deaths-of-despair"
config = load_config(PROJECT)
processed_dir = get_data_processed_dir(config)

state_panel = pd.read_parquet(processed_dir / "state_panel.parquet")
national_panel = pd.read_parquet(processed_dir / "national_panel.parquet")

STATE_INDICATORS = [
    "year", "state_fips", "state_name", "state_abbr",
    "overdose_death_rate", "suicide_rate", "deaths_despair_rate",
    "manufacturing_employees_thousands", "unemployment_rate",
    "median_household_income", "poverty_pct",
    "region", "mfg_yoy_pct_change", "mfg_index_2000",
]
available = [c for c in STATE_INDICATORS if c in state_panel.columns]
state_panel = state_panel[available].copy()

print(f"state_panel: {state_panel.shape[0]:,} rows × {state_panel.shape[1]} cols")
print(f"Years: {int(state_panel.year.min())} – {int(state_panel.year.max())}")
print(f"States: {state_panel.state_fips.nunique()}")
print()
print("national_panel:", national_panel.shape)
print(national_panel[["year","overdose_death_rate","suicide_rate","deaths_despair_rate"]].to_string(index=False))


state_panel: 918 rows × 14 cols
Years: 1999 – 2016
States: 51

national_panel: (18, 10)
 year  overdose_death_rate  suicide_rate  deaths_despair_rate
 1999             5.811276     11.762745            17.574022
 2000             6.353263     11.707843            18.061106
 2001             7.306953     12.127451            19.434404
 2002             8.408365     12.339216            20.747580
 2003             9.471647     12.215686            21.687333
 2004             9.834690     12.372549            22.207239
 2005            10.421622     12.307843            22.729465
 2006            11.947778     12.409804            24.357582
 2007            12.327122     12.658824            24.985945
 2008            12.681516     13.141176            25.822692
 2009            12.377300     13.149020            25.526320
 2010            13.012459     13.823529            26.835988
 2011            14.157086     13.970588            28.127675
 2012            14.048533     14.447059    

## Univariate profiles — key indicators

In [2]:
num_cols = ["overdose_death_rate","suicide_rate","deaths_despair_rate",
            "manufacturing_employees_thousands","unemployment_rate",
            "median_household_income","poverty_pct"]
available_num = [c for c in num_cols if c in state_panel.columns]
print(state_panel[available_num].describe().round(2).to_string())


       overdose_death_rate  suicide_rate  deaths_despair_rate  manufacturing_employees_thousands  unemployment_rate  median_household_income  poverty_pct
count               918.00        918.00               918.00                             882.00             774.00                   612.00       612.00
mean                 12.12         13.27                25.39                             273.98               5.66                 51798.70        14.05
std                   6.16          3.97                 8.37                             285.42               1.99                  9127.99         3.28
min                   1.82          3.80                 8.48                               8.74               2.16                 32938.00         7.08
25%                   7.99         10.80                19.16                              61.19               4.25                 45086.50        11.47
50%                  11.32         12.70                24.16               

## National trend — deaths of despair over time

In [3]:
nat = national_panel.dropna(subset=["deaths_despair_rate"]).sort_values("year")
fig = go.Figure()
fig.add_trace(go.Scatter(x=nat["year"], y=nat["overdose_death_rate"],
    mode="lines+markers", name="Drug Overdose Rate",
    line=dict(color="#e74c3c", width=3)))
fig.add_trace(go.Scatter(x=nat["year"], y=nat["suicide_rate"],
    mode="lines+markers", name="Suicide Rate",
    line=dict(color="#8e44ad", width=3)))
fig.add_trace(go.Scatter(x=nat["year"], y=nat["deaths_despair_rate"],
    mode="lines+markers", name="Combined Rate",
    line=dict(color="#2c3e50", width=3, dash="dash")))
fig.update_layout(
    title="Deaths of Despair Rate per 100k — National Average (1999–2016)",
    xaxis_title="Year", yaxis_title="Age-adjusted death rate per 100,000",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()


## Geographic variation — latest year snapshot

In [4]:
latest = state_panel[state_panel["year"] == state_panel["year"].max()].copy()
fig = px.choropleth(
    latest,
    locations="state_abbr",
    locationmode="USA-states",
    color="deaths_despair_rate",
    scope="usa",
    color_continuous_scale="Reds",
    hover_data={"state_name": True, "overdose_death_rate": ":.1f", "suicide_rate": ":.1f"},
    title=f"Deaths of Despair per 100k — {int(latest.year.iloc[0])}",
    labels={"deaths_despair_rate": "Deaths per 100k"},
)
fig.update_layout(height=500)
fig.show()
print("\nTop 10 states:")
print(latest.nlargest(10, "deaths_despair_rate")[
    ["state_abbr","overdose_death_rate","suicide_rate","deaths_despair_rate","region"]
].to_string(index=False))



Top 10 states:
state_abbr  overdose_death_rate  suicide_rate  deaths_despair_rate                 region
        WV              52.0211          19.3              71.3211             Appalachia
        NH              38.9902          17.2              56.1902                  Other
        OH              39.1225          14.2              53.3225 Rust Belt / Appalachia
        PA              37.8509          14.7              52.5509 Rust Belt / Appalachia
        KY              33.5499          16.8              50.3499             Appalachia
        NM              25.1877          22.5              47.6877                  Other
        ME              28.7144          15.9              44.6144                  Other
        UT              22.3516          21.8              44.1516                  Other
        DC              38.8000           5.2              44.0000                  Other
        NV              21.6704          21.4              43.0704                  

## Changepoint detection — when did it accelerate?

In [5]:
import sys
try:
    import ruptures as rpt
    nat_series = national_panel.dropna(subset=["overdose_death_rate"]).sort_values("year")
    signal = nat_series["overdose_death_rate"].values
    model = rpt.Pelt(model="rbf").fit(signal)
    breakpoints = model.predict(pen=5)
    years = nat_series["year"].values
    print("Detected changepoints at indices:", breakpoints)
    cp_years = [years[i-1] for i in breakpoints[:-1]]
    print("Changepoint years:", cp_years)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=nat_series["year"], y=signal, mode="lines+markers", name="Overdose rate"))
    for y in cp_years:
        fig.add_vline(x=y, line_dash="dash", line_color="red", annotation_text=str(int(y)))
    fig.update_layout(title="Overdose Death Rate with Changepoints", xaxis_title="Year", yaxis_title="Rate per 100k")
    fig.show()
except ImportError:
    print("ruptures not available — skipping changepoint detection")


Detected changepoints at indices: [18]
Changepoint years: []


## Correlation matrix — state-level indicators

In [6]:
corr_cols = [c for c in ["overdose_death_rate","suicide_rate","deaths_despair_rate",
    "manufacturing_employees_thousands","unemployment_rate",
    "median_household_income","poverty_pct"] if c in state_panel.columns]
corr = state_panel[corr_cols].corr().round(2)
fig = px.imshow(corr, text_auto=True, aspect="auto",
    color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
    title="Correlation Matrix — State-Level Indicators")
fig.show()
print(corr.to_string())


                                   overdose_death_rate  suicide_rate  deaths_despair_rate  manufacturing_employees_thousands  unemployment_rate  median_household_income  poverty_pct
overdose_death_rate                               1.00          0.33                 0.89                              -0.16               0.28                     0.04         0.22
suicide_rate                                      0.33          1.00                 0.72                              -0.44               0.06                    -0.23         0.11
deaths_despair_rate                               0.89          0.72                 1.00                              -0.32               0.24                    -0.08         0.22
manufacturing_employees_thousands                -0.16         -0.44                -0.32                               1.00               0.13                    -0.00         0.15
unemployment_rate                                 0.28          0.06                 0.24 

## Regional distribution — deaths of despair by region

In [7]:
if "region" in state_panel.columns:
    reg_latest = state_panel[state_panel["year"] == state_panel["year"].max()]
    fig = px.box(reg_latest, x="region", y="deaths_despair_rate",
        color="region", points="all",
        title=f"Deaths of Despair by Region — {int(reg_latest.year.iloc[0])}",
        labels={"deaths_despair_rate": "Deaths per 100k", "region": "Region"})
    fig.show()


## Manufacturing employment trend — national

In [8]:
mfg_nat = national_panel.dropna(subset=["manufacturing_employees_thousands"]).sort_values("year")
fig = go.Figure()
fig.add_trace(go.Scatter(x=mfg_nat["year"], y=mfg_nat["manufacturing_employees_thousands"],
    mode="lines+markers", name="Avg Mfg Employment (thousands)",
    line=dict(color="#2980b9", width=3)))
fig.update_layout(
    title="Average State Manufacturing Employment 1999–2016",
    xaxis_title="Year", yaxis_title="Thousands of workers",
)
fig.show()
